# Free particle on an oblique periodic cell

**Status:** Maintained package-backed numerical laboratory for a nondimensional free particle on one two-dimensional periodic primitive cell.

**Prerequisite:** [Nonorthogonal lattice metric geometry](../../../../../docs/lecturenotes/solidstate/lattices/nonorthogonal-metric-geometry/index.md) and the [laboratory note](../../../../../docs/lecturenotes/qm/piab2d/tise/numerical/oblique-periodic-free-particle/index.md).

The calculation uses $\hbar=m=1$. `periodic.finite_difference` owns the endpoint-excluded grid, inverse-metric stencil, mixed derivative, and quotient-seam Bloch phases. This notebook composes the geometric Laplacian into $H=-\Delta_h/2$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse.linalg import eigsh

from projectkoios.physkit.periodic.finite_difference.grid import PeriodicFiniteDifferenceGrid
from projectkoios.physkit.periodic.finite_difference.laplacian import BlochPeriodicLaplacianConstructionRequest, BlochPeriodicLaplacianConstructor
from projectkoios.physkit.periodic.lattice.boundary_phases import BoundaryTwistLift
from projectkoios.physkit.periodic.lattice.finite_domain import FinitePeriodicDomain, LatticeDimension
from projectkoios.physkit.periodic.lattice.lattice2d import DirectLattice2D
from projectkoios.physkit.periodic.lattice.metric import DirectLatticeMetric
from projectkoios.physkit.units import Unitless

direct_lattice = DirectLattice2D(
    a1=np.array([1.0, 0.0]),
    a2=np.array([0.45, 0.90]),
)
metric = DirectLatticeMetric(direct_lattice)

def construct_laplacian(shape, twist_turns):
    """Construct one package-owned metric Laplacian.

    Parameters
    ----------
    shape
        Endpoint-excluded fractional-axis point counts.
    twist_turns
        Bloch boundary twists in direct-axis turns.

    Returns
    -------
    BlochPeriodicLaplacian
        Correlated grid, twist fiber, and sparse represented operator.
    """
    grid = PeriodicFiniteDifferenceGrid(
        metric=metric,
        domain=FinitePeriodicDomain(LatticeDimension.TWO, shape),
        direct_basis_unit=Unitless(),
    )
    return BlochPeriodicLaplacianConstructor().action(
        request=BlochPeriodicLaplacianConstructionRequest(
            grid=grid,
            twist=BoundaryTwistLift(LatticeDimension.TWO, twist_turns),
        )
    )

def sampled_plane_wave(shape, mode, twist_turns):
    """Return one C-order sampled Bloch-compatible plane wave.

    Parameters
    ----------
    shape
        Fractional grid shape.
    mode
        Integer reciprocal-mode labels.
    twist_turns
        Fractional reciprocal shift in turns.

    Returns
    -------
    numpy.ndarray
        Complex sampled mode in last-axis-fastest order.
    """
    indices = np.indices(shape, dtype=np.float64)
    phase = np.zeros(shape, dtype=np.float64)
    for axis, extent in enumerate(shape):
        phase += (
            2.0 * np.pi * (mode[axis] + twist_turns[axis])
            * indices[axis] / extent
        )
    return np.exp(1j * phase).ravel(order="C")

def continuum_energy(mode, twist_turns):
    r"""Return the nondimensional continuum metric energy.

    Parameters
    ----------
    mode
        Integer reciprocal-mode labels.
    twist_turns
        Bloch shift in reciprocal turns.

    Returns
    -------
    float
        Energy for $\hbar=m=1$.
    """
    wave = 2.0 * np.pi * (
        np.asarray(mode, dtype=np.float64)
        + np.asarray(twist_turns, dtype=np.float64)
    )
    return 0.5 * float(wave @ metric.inverse_metric_tensor @ wave)


## Oblique geometry

For $\mathbf r=A\mathbf s$, the Laplacian is

$$\nabla^2=\sum_{ij}g^{ij}\partial_i\partial_j,$$

with $g=A^{\mathsf T}A$. The nonzero off-diagonal entry of $g^{-1}$ requires a mixed finite-difference stencil.


In [ ]:
assert np.allclose(metric.primitive_basis.T @ metric.reciprocal_basis, 2.0 * np.pi * np.eye(2))
assert abs(metric.inverse_metric_tensor[0, 1]) > 0.0
print("A =\n", metric.primitive_basis)
print("g =\n", metric.metric_tensor)
print("g^{-1} =\n", metric.inverse_metric_tensor)
print(f"cell area = {metric.measure:.6f}")

cell_fractional = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
cell_cartesian = metric.to_cartesian(cell_fractional)
fig, ax = plt.subplots(figsize=(5.8, 4.8))
ax.plot(cell_cartesian[:, 0], cell_cartesian[:, 1], marker="o")
ax.set(xlabel="x", ylabel="y", title="Oblique periodic primitive cell", aspect="equal")
ax.grid(alpha=0.25)
fig.tight_layout()


## Zero-twist sparse Hamiltonian

At zero twist, the constant mode must be the null mode of $\Delta_h$ and the ground state of $H=-\Delta_h/2$. The represented state uses one area weight $\Omega/(N_1N_2)$ per grid point.


In [ ]:
shape = (18, 20)
periodic = construct_laplacian(shape, (0.0, 0.0))
laplacian = periodic.represented_laplacian.to_csr()
hamiltonian = -0.5 * laplacian
hermitian_residual = hamiltonian - hamiltonian.conjugate().T
assert hermitian_residual.nnz == 0 or np.max(np.abs(hermitian_residual.data)) < 1e-12
assert np.linalg.norm(laplacian @ np.ones(periodic.request.grid.point_count)) < 1e-11
energies, eigenvectors = eigsh(hamiltonian, k=6, which="SA")
order = np.argsort(energies)
energies = energies[order]
eigenvectors = eigenvectors[:, order]
assert abs(energies[0]) < 1e-11

# A uniform fractional sample represents its equal share of the physical
# parallelogram area. This factor is required for state normalization.
area_weight = metric.measure / periodic.request.grid.point_count
ground_state = eigenvectors[:, 0]
ground_state = ground_state / np.sqrt(np.sum(np.abs(ground_state) ** 2) * area_weight)
assert np.isclose(np.sum(np.abs(ground_state) ** 2) * area_weight, 1.0)
print("lowest represented energies:", energies)


## Probability density on the physical parallelogram

A single free-particle plane wave has uniform density. The plot uses physical cell-edge coordinates rather than displaying the fractional array as a rectangular Cartesian box.


In [ ]:
edge_s0 = np.linspace(0.0, 1.0, shape[0] + 1)
edge_s1 = np.linspace(0.0, 1.0, shape[1] + 1)
edge_0, edge_1 = np.meshgrid(edge_s0, edge_s1, indexing="ij")
edge_fractional = np.stack((edge_0, edge_1), axis=-1)
edge_cartesian = metric.to_cartesian(edge_fractional)
density = np.abs(ground_state.reshape(shape, order="C")) ** 2
assert np.allclose(density, 1.0 / metric.measure, rtol=1e-11, atol=1e-11)
fig, ax = plt.subplots(figsize=(6.2, 5.0))
image = ax.pcolormesh(
    edge_cartesian[..., 0], edge_cartesian[..., 1], density,
    shading="flat", cmap="viridis",
)
ax.set(xlabel="x", ylabel="y", title="Normalized zero-twist ground-state density", aspect="equal")
fig.colorbar(image, ax=ax, label=r"$|\psi|^2$")
fig.tight_layout()


## Nonzero Bloch twist and discrete plane-wave identity

For twist turns $\boldsymbol\kappa$, sampled modes are $\exp[i2\pi(\mathbf m+\boldsymbol\kappa)\cdot\mathbf n/\mathbf N]$. The maintained quotient-seam operator must act on each mode with its exact discrete symbol and converge to

$$E_{\mathbf m}(\boldsymbol\kappa)=\frac12[2\pi(\mathbf m+\boldsymbol\kappa)]^{\mathsf T}g^{-1}[2\pi(\mathbf m+\boldsymbol\kappa)].$$


In [ ]:
twist_turns = (0.17, 0.29)
mode = (1, -1)
twisted = construct_laplacian(shape, twist_turns)
twisted_hamiltonian = -0.5 * twisted.represented_laplacian.to_csr()
twisted_residual = twisted_hamiltonian - twisted_hamiltonian.conjugate().T
assert twisted_residual.nnz == 0 or np.max(np.abs(twisted_residual.data)) < 1e-12
# Verify the complete sparse operator, including mixed-derivative corner
# wraps, against its exact finite-difference plane-wave eigenvalue.
mode_vector = sampled_plane_wave(shape, mode, twist_turns)
observed_energy = float(
    np.vdot(mode_vector, twisted_hamiltonian @ mode_vector).real
    / np.vdot(mode_vector, mode_vector).real
)
inverse_metric = metric.inverse_metric_tensor
theta = 2.0 * np.pi * (np.asarray(mode) + np.asarray(twist_turns)) / np.asarray(shape)
spacings = 1.0 / np.asarray(shape, dtype=np.float64)
discrete_laplacian = 0.0
for axis in range(2):
    discrete_laplacian -= 4.0 * inverse_metric[axis, axis] * np.sin(theta[axis] / 2.0) ** 2 / spacings[axis] ** 2
discrete_laplacian -= (
    2.0 * inverse_metric[0, 1] * np.sin(theta[0]) * np.sin(theta[1])
    / (spacings[0] * spacings[1])
)
expected_discrete_energy = -0.5 * discrete_laplacian
assert np.isclose(observed_energy, expected_discrete_energy, rtol=2e-13, atol=2e-13)
print(f"discrete energy = {observed_energy:.8f}")
print(f"continuum energy = {continuum_energy(mode, twist_turns):.8f}")


## Second-order refinement

The centered metric stencil should approach the continuum plane-wave energy with second-order error for a fixed mode and twist.


In [ ]:
extents = np.array([8, 16, 32])
continuum_reference = continuum_energy(mode, twist_turns)
errors = []
for extent in extents:
    refinement_shape = (int(extent), int(extent))
    represented = construct_laplacian(refinement_shape, twist_turns)
    represented_hamiltonian = -0.5 * represented.represented_laplacian.to_csr()
    vector = sampled_plane_wave(refinement_shape, mode, twist_turns)
    energy = float(
        np.vdot(vector, represented_hamiltonian @ vector).real
        / np.vdot(vector, vector).real
    )
    errors.append(abs(energy - continuum_reference))
errors = np.asarray(errors)
ratios = errors[:-1] / errors[1:]
assert np.all(ratios > 3.5)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.loglog(extents, errors, marker="o", label="observed")
ax.loglog(extents, errors[0] * (extents[0] / extents) ** 2, linestyle="--", label=r"$N^{-2}$ reference")
ax.set(xlabel="Points per fractional axis", ylabel="Plane-wave energy error", title="Second-order metric-stencil refinement")
ax.grid(which="both", alpha=0.25)
ax.legend()
fig.tight_layout()


## Interpretation and exercises

The sparse eigenspectrum verifies the represented low modes. Plane-wave identities and refinement provide independent numerical checks. The model covers a nondimensional scalar free particle on a constant-metric periodic cell.

1. Set the skew component of $\mathbf a_2$ to zero and recover the orthogonal Kronecker-sum spectrum.
2. Change the twist by an integer vector and verify spectral invariance while retaining the unreduced lift.
3. Compare several fixed integer modes under independent refinement of the two fractional axes.
4. Repeat with physical length units and compose the appropriate $\hbar^2/(2m)$ conversion explicitly.
5. Explain why a single plane wave has uniform probability density even on an oblique displayed cell.
